# CDVAEのMEGNet

In [44]:
import torch
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
import glob

%matplotlib inline

In [45]:
result_dir = '/home/fujii/cdvae_comparison/main_results/megnet/'
os.makedirs(result_dir, exist_ok=True)

## CDVAEモデルの学習結果
- lr=0.00100に決定

In [46]:
api = wandb.Api()

In [4]:
urls = [
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/ukyvgors',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/171p2zrl',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/czr955zi',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/13w8iowr',

]
lrs = [
    5e-4,
    1e-4,
    1e-3,
    1e-5,
]

In [5]:
all_results = []
for lr, url in zip(lrs, urls):
    run = api.run(url)
    # ログされた履歴を取得
    history = run.history()

    # val_lossの最小値とそのステップを取得
    min_step = history['val_loss'].idxmin()
    min_val_loss = history['val_loss'][min_step]
    val_natom_loss = history['val_natom_loss'][min_step]
    val_natom_accuracy = history['val_natom_accuracy'][min_step]
    val_lattice_loss = history['val_lattice_loss'][min_step]
    val_eform_mae = history['val_eform_mae'][min_step]
    val_gap_mae = history['val_gap_mae'][min_step]
    val_tolerance_acc = history['val_tolerance_acc'][min_step]
    val_100more_acc = history['val_100more_acc'][min_step]

    all_results.append({
        'lr': lr,
        'min_val_loss': min_val_loss,
        'val_natom_loss': val_natom_loss,
        'val_natom_accuracy': val_natom_accuracy,
        'val_lattice_loss': val_lattice_loss,
        'val_eform_mae': val_eform_mae,
        'val_gap_mae': val_gap_mae,
        'val_tolerance_acc': val_tolerance_acc,
        'val_100more_acc': val_100more_acc,
    })

In [6]:
df = pd.DataFrame(all_results)
df.to_csv(os.path.join(result_dir, 'forward_result.csv'), index=False)
display(df)

,lr,min_val_loss,val_natom_loss,val_natom_accuracy,val_lattice_loss,val_eform_mae,val_gap_mae,val_tolerance_acc,val_100more_acc
0,0.00050,16.157167,5.588017,0.357912,0.713992,0.311164,0.883223,0.994828,1.0
1,0.00010,15.433899,5.304371,0.386220,0.652582,0.271822,0.877551,0.995901,1.0
2,0.00100,14.841366,5.049627,0.338767,0.650436,0.267936,0.874966,0.994828,1.0
3,0.00001,22.913857,2.119287,0.352837,0.692602,0.403935,0.900415,0.996878,1.0


## 最適化推論の学習率調整
- 推論コマンドの例: 
    - ```poetry run python scripts/evaluate.py --tasks opt --label lr00001 --lr 0.0001 --num_starting_points 128 --model_path /home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/shin_megnet_lr5e-6/ --target_bg 2.5 --num_saved_crys 0 --megnet_loss_mode True --coef_e_form 0 ```

In [115]:
result_dir = '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/shin_megnet_lr5e-6/'
#os.listdir(result_dir)

In [126]:
bandgap_margin = 0.2
raw_result_dict_list = []
decoded_result_dict_list = []
result_dict_list = []
for pred_path in glob.glob(os.path.join(result_dir,'eval_opt_megnet__bg2.50__*.pt')):
    d = torch.load(pred_path)
    d.keys()
    # 最適化に使った学習率の情報を得る
    lr_string = pred_path.split("eval_opt_lr")[-1].split(".")[0]
    #lr = float("0."+lr_string[1:])

    # ファイル名から学習率などを取得
    tag_dict = {}
    tag_name_list = ['lr','bg','grad-steps']
    for tag_data in os.path.basename(pred_path).split("__"):
        for tag_name in tag_name_list:
            if tag_data.startswith(tag_name):
                if tag_name == 'grad-steps':
                    tag_dict[tag_name] = int(tag_data.replace(tag_name,"").replace('.pt',''))
                else:
                    tag_dict[tag_name] = float(tag_data.replace(tag_name,"").replace('.pt',''))

    # band gap の予測値を得る
    raw_z_bandgap_pred = d['prediction_matched'][:,0]
    decoded_z_bandgap_pred = d['prediction_decoded'][:,0]
    assert raw_z_bandgap_pred.shape == decoded_z_bandgap_pred.shape
    ## 128になるまで、torch.nanを追加する
    if raw_z_bandgap_pred.shape[0] < 128:
        raw_z_bandgap_pred = torch.cat([raw_z_bandgap_pred, torch.full((128-raw_z_bandgap_pred.shape[0],), float('nan'))])
    if decoded_z_bandgap_pred.shape[0] < 128:
        decoded_z_bandgap_pred = torch.cat([decoded_z_bandgap_pred, torch.full((128-decoded_z_bandgap_pred.shape[0],), float('nan'))])
    # 予測バンドギャップとターゲットの差分をとる
    raw_z_bg_satisfaction = torch.clip(torch.abs(raw_z_bandgap_pred - tag_dict['bg'])-bandgap_margin, min=0).numpy() == 0
    decoded_z_bg_satisfaction = torch.clip(torch.abs(decoded_z_bandgap_pred - tag_dict['bg'])-bandgap_margin, min=0).numpy() == 0
    result_dict_list.append({
        'lr': tag_dict['lr'],
        'bg': tag_dict['bg'],
        'bg-margin': bandgap_margin,
        'grad_steps': tag_dict['grad-steps'],
        'raw_z_bg_satisfaction_rate': raw_z_bg_satisfaction.sum()/raw_z_bg_satisfaction.shape[0],
        'decoded_z_bg_satisfaction_rate': decoded_z_bg_satisfaction.sum()/decoded_z_bg_satisfaction.shape[0],
    })
result_df = pd.DataFrame(result_dict_list).set_index(['bg','bg-margin','lr','grad_steps']).sort_index()
result_df

/tmp/ipykernel_835105/708758115.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(pred_path)


raw_z_bg_satisfaction_rate  \
bg  bg-margin lr      grad_steps                               
2.5 0.2       0.00001 200                           0.015625   
                      400                           0.007812   
                      800                           0.015625   
                      1600                          0.031250   
                      3200                          0.046875   
                      5000                          0.054688   
              0.00010 200                           0.054688   
                      400                           0.039062   
                      800                           0.078125   
                      1600                          0.132812   
                      3200                          0.273438   
                      5000                          0.273438   
              0.00100 200                           0.164062   
                      400                           0.281250   
                      800                           0.289062   
                      1600                          0.281250   
                      3200                          0.281250   
                      5000                          0.296875   
              0.01000 200                           0.289062   
                      400                           0.281250   
                      800                           0.304688   
                      1600                          0.289062   
                      3200                          0.281250   
                      5000                          0.289062   

                                  decoded_z_bg_satisfaction_rate  
bg  bg-margin lr      grad_steps                                  
2.5 0.2       0.00001 200                               0.007812  
                      400                               0.000000  
                      800                               0.007812  
                      1600                              0.007812  
                      3200                              0.000000  
                      5000                              0.007812  
              0.00010 200                               0.007812  
                      400                               0.000000  
                      800                               0.000000  
                      1600                              0.007812  
                      3200                              0.023438  
                      5000                              0.000000  
              0.00100 200                               0.007812  
                      400                               0.000000  
                      800                               0.000000  
                      1600                              0.000000  
                      3200                              0.000000  
                      5000                              0.000000  
              0.01000 200                               0.007812  
                      400                               0.007812  
                      800                               0.007812  
                      1600                              0.015625  
                      3200                              0.007812  
                      5000                              0.000000

In [130]:
# best result
display(result_df.sort_values('decoded_z_bg_satisfaction_rate',ascending=False).head(1))
bg, margin, best_lr, best_steps = result_df.sort_values('decoded_z_bg_satisfaction_rate',ascending=False).head(1).index.values[0]

,,,,raw_z_bg_satisfaction_rate,decoded_z_bg_satisfaction_rate
bg,bg-margin,lr,grad_steps,,
2.5,0.2,0.0001,3200,0.273438,0.023438


(2.5, 0.2, 0.0001, 3200)